# LSTM Workload Forecasting (One Fit per Trace)

This notebook trains LSTM once on each of the 12 processed traces. That is **12 fits total**. Each fit is a single bivariate model that predicts CPU and memory together, using that trace's training split. Each run has a 500-epoch maximum and early stopping (patience 20), so training can stop sooner. Input windows are 288 steps and the forecast horizon is 2 steps.

This is a reduced comparison, not a full reproduction of the paper: it uses one seed and only the single-training bivariate (S-B) setup. It reports MSE/MAE and service metrics (SR, TPR) per trace. The paper's S-B values are included as context; because the paper aggregates more configurations and repeated runs, treat comparisons as descriptive.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = "https://github.com/grass-overflow/ML_Proj.git"
repo_directory = Path("/kaggle/working/ML_Proj")
if (repo_directory / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_directory), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_directory)], check=True)
os.chdir(repo_directory)
missing = [name for name in ("requirements-cpu.txt", "run_experiments.py") if not Path(name).is_file()]
if missing:
    raise RuntimeError("Repository checkout is missing: " + ", ".join(missing))
print("Repository ready at", Path.cwd())

### Install Kaggle-Compatible Dependencies

Kaggle's current Python 3.13 image cannot install this project's pinned TensorFlow 2.16.2. On Python 3.13, reuse and validate Kaggle's preinstalled TensorFlow/TensorFlow Probability stack. On Python 3.12 or older, remove conflicting JAX packages and install the pinned CPU requirements.

In [ ]:
import importlib.metadata
import subprocess
import sys

if sys.version_info >= (3, 13):
    import tensorflow as tf
    import tensorflow_probability as tfp
    import tf_keras
    print("Using Kaggle-provided TensorFlow", importlib.metadata.version("tensorflow"))
    print("TensorFlow Probability", importlib.metadata.version("tensorflow-probability"))
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cpu.txt"], check=True)

### Run one LSTM fit for each of the 12 traces

In Kaggle Notebook settings, select **GPU** (not TPU), restart the session, and run all cells from the top. For each of the 12 traces, the runner trains one bivariate model (CPU and memory together), with a 500-epoch cap and early stopping at patience 20. This is 12 fits total. The final cell exports this run's point metrics, service-level/TPR metrics, and supporting metrics as model-prefixed CSV files in the downloadable ZIP.

In [ ]:
import csv
import os
import subprocess
import sys
from pathlib import Path
import tensorflow as tf

visible_gpus = tf.config.list_physical_devices("GPU")
if not visible_gpus:
    raise RuntimeError("No GPU is visible. In Kaggle, set Accelerator to GPU, restart the session, and run all cells again.")
print("Using GPU:", visible_gpus)
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["WORKLOAD_USE_GPU"] = "1"
model_name = "LSTM"
datasets = ["ali18","ali20_c","ali20_g","gc11","gc19_a","gc19_b","gc19_c","gc19_d","gc19_e","gc19_f","gc19_g","gc19_h"]
metrics_path = Path("results/raw/model_metrics.csv")
existing_metric_rows = 0
if metrics_path.is_file():
    with metrics_path.open(newline="", encoding="utf-8") as source:
        existing_metric_rows = max(sum(1 for _ in csv.reader(source)) - 1, 0)
print(f"Training {model_name} once per trace ({len(datasets)} fits total):", datasets)
common = ["--models", model_name, "--runs", "1", "--epochs", "500", "--patience", "20", "--paper-faithful", "--mc-samples", "30", "--inference-repeats", "10", "--no-plots"]
for index, dataset in enumerate(datasets, 1):
    print(f"[{index}/{len(datasets)}] One bivariate fit: {model_name} on {dataset}", flush=True)
    subprocess.run([sys.executable, "run_experiments.py", "--mode", "runtime", "--datasets", dataset] + common, check=True)
print("Training complete; expected 12 fits and 24 resource metric rows.")

### Per-Trace Metrics and Plots

The runner records point-error and service-level metrics for each trace. This cell exports the model metrics plus service-level, quantile, runtime, and uncertainty CSVs using model-prefixed filenames. The plots summarize MSE/MAE across all 12 traces; the separate comparison notebook combines the three models to produce paper-style SR/TPR comparisons.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

Path("results/plots").mkdir(parents=True, exist_ok=True)
layers = ["288-step input", "1D convolution", "LSTM layer", "Dense layers", "Point forecast"]
fig, axis = plt.subplots(figsize=(6, 7))
axis.set_xlim(0, 1)
axis.set_ylim(0, len(layers) + 0.5)
axis.axis("off")
for index, label in enumerate(layers):
    y = len(layers) - index - 0.5
    axis.add_patch(FancyBboxPatch((0.12, y - 0.22), 0.76, 0.42, boxstyle="round,pad=0.02", facecolor="#e9f1f5", edgecolor="#315b6d"))
    axis.text(0.5, y, label, ha="center", va="center")
    if index < len(layers) - 1:
        axis.annotate("", (0.5, y - 0.31), (0.5, y - 0.08), arrowprops={"arrowstyle": "->"})
axis.set_title("LSTM architecture")
fig.tight_layout()
fig.savefig("results/plots/lstm_architecture.png", dpi=150)
display(fig)
plt.close(fig)

fig, axis = plt.subplots(figsize=(10, 3))
axis.axis("off")
workflow = ["Historical workload", "Predictive model", "Resource manager", "Cloud resources"]
positions = [0.12, 0.38, 0.64, 0.88]
for x, label in zip(positions, workflow):
    axis.text(x, 0.55, label, ha="center", va="center", bbox={"boxstyle": "round,pad=0.45", "facecolor": "#e9f1f5", "edgecolor": "#315b6d"})
for left, right in zip(positions, positions[1:]):
    axis.annotate("", (right - 0.08, 0.55), (left + 0.08, 0.55), arrowprops={"arrowstyle": "->"})
axis.annotate("updated history", (0.12, 0.2), (0.88, 0.2), arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=-0.25"}, ha="center")
axis.set_title("Cloud workload prediction workflow")
fig.tight_layout()
fig.savefig("results/plots/workload_prediction_workflow.png", dpi=150)
display(fig)
plt.close(fig)

In [ ]:
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, display

if "existing_metric_rows" not in globals():
    raise RuntimeError("Run the training cell first; it records the starting row to exclude older results.")
raw_path = Path("results/raw/model_metrics.csv")
if not raw_path.is_file():
    raise FileNotFoundError(f"Expected runner output was not found: {raw_path}")
raw = pd.read_csv(raw_path)
run_metrics = raw.iloc[existing_metric_rows:].copy()
run_metrics = run_metrics[run_metrics["model"].astype(str).str.endswith("-" + model_name)]
run_metrics = run_metrics[run_metrics["dataset"].isin(datasets)]
expected_pairs = {(dataset, resource) for dataset in datasets for resource in ("avgcpu", "avgmem")}
actual_pairs = set(zip(run_metrics["dataset"], run_metrics["resource"]))
if actual_pairs != expected_pairs or len(run_metrics) != len(expected_pairs):
    raise RuntimeError(f"Expected one CPU and one memory result for each of the 12 traces ({len(expected_pairs)} rows); found {len(run_metrics)} rows.")

result_dir = Path("results/representative_runs") / model_name
plot_dir = result_dir / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
paper = {"avgcpu":[0.0047,0.05],"avgmem":[0.0054,0.0487]}
comparison = run_metrics[["model", "dataset", "resource", "MSE", "MAE", "training_time", "epochs_requested"]].copy()
comparison["paper_S-B_MSE_reference"] = comparison["resource"].map({"avgcpu": paper["avgcpu"][0], "avgmem": paper["avgmem"][0]})
comparison["paper_S-B_MAE_reference"] = comparison["resource"].map({"avgcpu": paper["avgcpu"][1], "avgmem": paper["avgmem"][1]})
comparison.to_csv(result_dir / f"{model_name}_model_metrics.csv", index=False)
display(comparison)

run_ids = set(run_metrics["run_id"].dropna().astype(str))
for metric_name in ("quantile_metrics", "service_level_metrics", "runtime_metrics", "uncertainty_metrics", "uncertainty_summary"):
    metric_path = Path("results/raw") / f"{metric_name}.csv"
    if metric_path.is_file():
        metric_frame = pd.read_csv(metric_path)
        if "run_id" not in metric_frame.columns:
            raise RuntimeError(f"Cannot isolate this run's {metric_name}; the CSV has no run_id column.")
        metric_frame = metric_frame[metric_frame["run_id"].astype(str).isin(run_ids)]
        metric_frame.to_csv(result_dir / f"{model_name}_{metric_name}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, metric, reference in zip(axes, ("MSE", "MAE"), ("paper_S-B_MSE_reference", "paper_S-B_MAE_reference")):
    run_metrics.pivot(index="dataset", columns="resource", values=metric).reindex(datasets).plot(kind="bar", ax=axis, color={"avgcpu": "#287c8e", "avgmem": "#e07a5f"})
    axis.axhline(paper["avgcpu"][0 if metric == "MSE" else 1], color="#287c8e", linestyle="--", label="Paper S-B CPU reference")
    axis.axhline(paper["avgmem"][0 if metric == "MSE" else 1], color="#e07a5f", linestyle=":", label="Paper S-B memory reference")
    axis.set_title(f"LSTM: {metric} per trace")
    axis.set_ylabel(metric)
    axis.set_xlabel("Trace")
    axis.legend(fontsize=8)
    axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
metrics_plot = plot_dir / "metrics_by_trace.png"
fig.savefig(metrics_plot, dpi=150)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(len(datasets), 2, figsize=(12, 4 * len(datasets)))
for row, dataset in enumerate(datasets):
    frame = pd.read_csv(Path("saved_data/preprocessed") / f"{dataset}.csv")
    for col, resource in enumerate(("avgcpu", "avgmem")):
        axis = axes[row, col]
        axis.hist(frame[resource].dropna(), bins=40, color="#287c8e" if col == 0 else "#e07a5f", alpha=0.85)
        axis.set_title(f"{dataset}: {resource}")
        axis.set_xlabel("Normalized utilization")
        axis.set_ylabel("Frequency")
fig.tight_layout()
distribution_plot = plot_dir / "selected_trace_distributions.png"
fig.savefig(distribution_plot, dpi=150)
display(fig)
plt.close(fig)

print(f"Run-specific metrics and plots: {result_dir}")
archive = shutil.make_archive(str(result_dir), "zip", root_dir=result_dir)
print(archive)
display(FileLink(archive))